# Student Habits vs Academic Performance — MLflow Experiment Tracking

This notebook trains two regression models on the **Student Habits vs Academic Performance** dataset and tracks every experiment with MLflow.

**Models trained:**
- Experiment 1 — Ridge Regression (linear baseline, multiple `alpha` values)
- Experiment 2 — Random Forest Regressor (ensemble, multiple `n_estimators` values)

**Dataset:** https://www.kaggle.com/datasets/jayaantanaath/student-habits-vs-academic-performance

## 1. Load Data

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

# Download the dataset
kagglehub.dataset_download("jayaantanaath/student-habits-vs-academic-performance")

# Load into a DataFrame
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "jayaantanaath/student-habits-vs-academic-performance",
    "student_habits_performance.csv",
)

df.head()

In [ ]:
# Dataset shape
print("Shape:", df.shape)

In [ ]:
# Is the target balanced?
df['exam_score'].describe()

## 2. Check for Missing Values

In [ ]:
# Check for missing values in each column
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

# Total missing values in the dataset
total_missing = df.isnull().sum().sum()
print(f"\nTotal missing values: {total_missing}")

# Summary of data types and non-null counts
df.info()

In [ ]:
df.describe()

## 3. Preprocessing

In [ ]:
# Fill missing parental_education_level with the column mode
df['parental_education_level'] = df['parental_education_level'].fillna(
    df['parental_education_level'].mode()[0]
)

# Confirm no more missing values
print("Missing values after fill:")
print(df.isnull().sum())

In [ ]:
# Ordinal-encode categorical columns
gender_map = {"Male": 0, "Female": 1, "Other": 2}
diet_map   = {"Poor": 0, "Fair": 1, "Good": 2}
par_ed_map = {"High School": 0, "Bachelor": 1, "Master": 2}
internet_map = {"Poor": 0, "Average": 1, "Good": 2}
clubs_map  = {"No": 0, "Yes": 1}
job_map    = {"No": 0, "Yes": 1}

df["gender"]                       = df["gender"].str.strip().str.title().map(gender_map)
df["diet_quality"]                  = df["diet_quality"].str.strip().str.title().map(diet_map)
df["parental_education_level"]      = df["parental_education_level"].str.strip().str.title().map(par_ed_map)
df["internet_quality"]              = df["internet_quality"].str.strip().str.title().map(internet_map)
df["extracurricular_participation"] = df["extracurricular_participation"].str.strip().str.title().map(clubs_map)
df["part_time_job"]                 = df["part_time_job"].str.strip().str.title().map(job_map)

# Drop the student_id column — it carries no predictive signal
df = df.drop(columns=["student_id"], errors="ignore")

df.head()

In [ ]:
df.columns

## 4. Train / Validation / Test Split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
TARGET = "exam_score"
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Bin target for stratified splitting (keeps score distributions balanced)
y_binned = pd.cut(y, bins=5, labels=False)

# First split: 60 % train, 40 % temp
X_train, X_temp, y_train, y_temp, yb_train, yb_temp = train_test_split(
    X, y, y_binned, test_size=0.40, random_state=42, stratify=y_binned
)

# Second split: 20 % validation, 20 % test from temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=yb_temp
)

print(f"Train set:      {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")

## 5. Scale Features

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

# Fit scaler on training data only
scaler = StandardScaler()
scaler.fit(X_train)

# Save the scaler for use in the prediction pipeline
joblib.dump(scaler, 'scaler.joblib')
print("Scaler saved to scaler.joblib")

# Transform all splits
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

## 6. MLflow Setup

In [ ]:
import os
import mlflow
import mlflow.sklearn

# Create mlruns directory
os.makedirs("./mlruns", exist_ok=True)

# Set up local tracking URI — creates a 'mlruns' folder in the current directory
mlflow.set_tracking_uri("file:./mlruns")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

---
## 7. Experiment 1 — Ridge Regression

We sweep over multiple `alpha` regularisation values and log each as a separate MLflow run.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set experiment
mlflow.set_experiment("Student_Habits_Ridge_Regression")

alphas = [0.1, 1.0, 10.0, 100.0]

for alpha in alphas:
    with mlflow.start_run(run_name=f"Ridge_alpha_{alpha}"):

        # --- Train ---
        model = Ridge(alpha=alpha, random_state=42)
        model.fit(X_train_s, y_train)

        # --- Predict ---
        y_val_pred  = model.predict(X_val_s)
        y_test_pred = model.predict(X_test_s)

        # --- Metrics ---
        val_rmse  = float(np.sqrt(mean_squared_error(y_val,  y_val_pred)))
        val_mae   = float(mean_absolute_error(y_val,  y_val_pred))
        val_r2    = float(r2_score(y_val,  y_val_pred))

        test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_pred)))
        test_mae  = float(mean_absolute_error(y_test, y_test_pred))
        test_r2   = float(r2_score(y_test, y_test_pred))

        # --- Log parameters ---
        mlflow.log_param("model_type",   "Ridge")
        mlflow.log_param("alpha",        alpha)
        mlflow.log_param("random_state", 42)

        # --- Log metrics ---
        mlflow.log_metric("val_rmse",  val_rmse)
        mlflow.log_metric("val_mae",   val_mae)
        mlflow.log_metric("val_r2",    val_r2)
        mlflow.log_metric("test_rmse", test_rmse)
        mlflow.log_metric("test_mae",  test_mae)
        mlflow.log_metric("test_r2",   test_r2)

        # --- Log model ---
        mlflow.sklearn.log_model(model, "model")

        # --- Capture run ID for later registration ---
        run_id = mlflow.active_run().info.run_id

        print(f"alpha={alpha:<6}  val_r2={val_r2:.4f}  test_rmse={test_rmse:.4f}  run_id={run_id}")

    # Register each run under its own model name
    mlflow.register_model(f"runs:/{run_id}/model", f"student_ridge_alpha_{alpha}")

### Find the best Ridge run

In [ ]:
# Find the best Ridge run (lowest test RMSE)
ridge_runs = mlflow.search_runs(
    experiment_names=["Student_Habits_Ridge_Regression"],
    order_by=["metrics.test_rmse ASC"]
)

best_ridge_run = ridge_runs.iloc[0]
best_ridge_run_id    = best_ridge_run["run_id"]
best_ridge_alpha     = best_ridge_run["params.alpha"]
best_ridge_test_rmse = best_ridge_run["metrics.test_rmse"]
best_ridge_test_r2   = best_ridge_run["metrics.test_r2"]

print(f"Best Ridge Model:")
print(f"  alpha     = {best_ridge_alpha}")
print(f"  test_rmse = {best_ridge_test_rmse:.4f}")
print(f"  test_r2   = {best_ridge_test_r2:.4f}")

# Register best Ridge model
mlflow.register_model(f"runs:/{best_ridge_run_id}/model", "best_student_ridge")
print("\nBest Ridge model registered as 'best_student_ridge'")

---
## 8. Experiment 2 — Random Forest Regressor

We sweep over multiple `n_estimators` values to find the best ensemble size.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Set experiment
mlflow.set_experiment("Student_Habits_Random_Forest")

n_estimators_list = [50, 100, 200, 300]

for n_est in n_estimators_list:
    with mlflow.start_run(run_name=f"RF_n_estimators_{n_est}"):

        # --- Train ---
        model = RandomForestRegressor(
            n_estimators=n_est,
            max_depth=10,
            min_samples_leaf=4,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train_s, y_train)

        # --- Predict ---
        y_val_pred  = model.predict(X_val_s)
        y_test_pred = model.predict(X_test_s)

        # --- Metrics ---
        val_rmse  = float(np.sqrt(mean_squared_error(y_val,  y_val_pred)))
        val_mae   = float(mean_absolute_error(y_val,  y_val_pred))
        val_r2    = float(r2_score(y_val,  y_val_pred))

        test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_pred)))
        test_mae  = float(mean_absolute_error(y_test, y_test_pred))
        test_r2   = float(r2_score(y_test, y_test_pred))

        # --- Log parameters ---
        mlflow.log_param("model_type",      "RandomForest")
        mlflow.log_param("n_estimators",    n_est)
        mlflow.log_param("max_depth",       10)
        mlflow.log_param("min_samples_leaf", 4)
        mlflow.log_param("random_state",    42)

        # --- Log metrics ---
        mlflow.log_metric("val_rmse",  val_rmse)
        mlflow.log_metric("val_mae",   val_mae)
        mlflow.log_metric("val_r2",    val_r2)
        mlflow.log_metric("test_rmse", test_rmse)
        mlflow.log_metric("test_mae",  test_mae)
        mlflow.log_metric("test_r2",   test_r2)

        # --- Log model ---
        mlflow.sklearn.log_model(model, "model")

        # --- Capture run ID for later registration ---
        run_id = mlflow.active_run().info.run_id

        print(f"n_estimators={n_est:<4}  val_r2={val_r2:.4f}  test_rmse={test_rmse:.4f}  run_id={run_id}")

    # Register each run under its own model name
    mlflow.register_model(f"runs:/{run_id}/model", f"student_rf_{n_est}_trees")

### Find the best Random Forest run

In [ ]:
# Find the best Random Forest run (lowest test RMSE)
rf_runs = mlflow.search_runs(
    experiment_names=["Student_Habits_Random_Forest"],
    order_by=["metrics.test_rmse ASC"]
)

best_rf_run = rf_runs.iloc[0]
best_rf_run_id    = best_rf_run["run_id"]
best_rf_n_est     = best_rf_run["params.n_estimators"]
best_rf_test_rmse = best_rf_run["metrics.test_rmse"]
best_rf_test_r2   = best_rf_run["metrics.test_r2"]

print(f"Best Random Forest Model:")
print(f"  n_estimators = {best_rf_n_est}")
print(f"  test_rmse    = {best_rf_test_rmse:.4f}")
print(f"  test_r2      = {best_rf_test_r2:.4f}")

# Register best RF model
mlflow.register_model(f"runs:/{best_rf_run_id}/model", "best_student_rf")
print("\nBest Random Forest model registered as 'best_student_rf'")

---
## 9. Compare Models & Register the Overall Best

In [ ]:
print("=" * 50)
print("Model Comparison — Test Set")
print("=" * 50)
print(f"Ridge Regression   test_rmse={best_ridge_test_rmse:.4f}  test_r2={best_ridge_test_r2:.4f}")
print(f"Random Forest      test_rmse={best_rf_test_rmse:.4f}  test_r2={best_rf_test_r2:.4f}")

if best_rf_test_rmse < best_ridge_test_rmse:
    overall_best_run_id = best_rf_run_id
    overall_best_label  = f"Random Forest (n_estimators={best_rf_n_est})"
else:
    overall_best_run_id = best_ridge_run_id
    overall_best_label  = f"Ridge Regression (alpha={best_ridge_alpha})"

print(f"\n✅ Overall best model: {overall_best_label}")

# Register the overall best model
mlflow.register_model(f"runs:/{overall_best_run_id}/model", "BestStudentHabitsModel")
print("Overall best model registered as 'BestStudentHabitsModel'")

## 10. Transition Best Model to Staging

In [ ]:
from mlflow.tracking import MlflowClient
from datetime import datetime

client = MlflowClient()
date = datetime.today().strftime("%Y-%m-%d")

model_version = 1
new_stage     = "Staging"

client.transition_model_version_stage(
    name="BestStudentHabitsModel",
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

print(f"Model 'BestStudentHabitsModel' version {model_version} transitioned to {new_stage}")

# Update the model version description
client.update_model_version(
    name="BestStudentHabitsModel",
    version=model_version,
    description=(
        f"Best model from Student Habits experiments: {overall_best_label}. "
        f"test_rmse={min(best_ridge_test_rmse, best_rf_test_rmse):.4f} as of {date}"
    )
)

print("Model description updated.")

## 11. Load Best Model from Registry & Evaluate

In [ ]:
import mlflow.pyfunc
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the best model from the model registry (Staging)
model_uri    = "models:/BestStudentHabitsModel/Staging"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Make predictions on the test set
predictions = loaded_model.predict(X_test_s)

# Display first 10 predictions vs actuals
print("First 10 predictions vs actual exam scores:")
for pred, actual in zip(predictions[:10], y_test.values[:10]):
    print(f"  predicted={pred:.2f}   actual={actual:.2f}")

# Final evaluation
rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
mae  = float(mean_absolute_error(y_test, predictions))
r2   = float(r2_score(y_test, predictions))

print(f"\nFinal Test Metrics (loaded from registry):")
print(f"  RMSE : {rmse:.4f}")
print(f"  MAE  : {mae:.4f}")
print(f"  R²   : {r2:.4f}")

## 12. Save Best Model Locally for FastAPI

The prediction pipeline (`predict_pipeline.py`) loads `best_model.joblib` at startup.

In [ ]:
import joblib
import mlflow.sklearn

# Load the sklearn flavour directly so we can call .predict() natively
best_sklearn_model = mlflow.sklearn.load_model(f"runs:/{overall_best_run_id}/model")

# Save for the FastAPI service
joblib.dump(best_sklearn_model, "best_model.joblib")
print("best_model.joblib saved — ready for predict_pipeline.py")